# 04. TF Multi-Input Embedding Modeling

`transactions.csv` 내부 정보만 사용해 단지/법정동/구 ID embedding이 03 선형 baseline보다 가격 예측 성능을 개선하는지 검증합니다.

- 데이터 정책: Policy B (`is_cancelled == 0`, `trade_type in [중개거래, unknown]`)
- target: `log(price_per_m2)`
- baseline: 03 full run의 Policy B LinearRegression/Ridge 중 valid `log_mae`가 낮은 모델
- 기본 실행: `RUN_MODE = "smoke"`

## 1. 환경 확인

In [ ]:
from pathlib import Path
import json
import math
import platform
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from tensorflow import keras

from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=FutureWarning)

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("tensorflow:", tf.__version__)
print("keras:", getattr(keras, "__version__", "unknown"))

## 2. 경로와 실행 설정

`RUN_MODE`만 바꾸면 smoke run과 full run을 같은 코드로 실행합니다. tracked notebook의 기본값은 smoke로 둡니다.

In [ ]:
# 1) 경로와 실행 설정
current_dir = Path.cwd()
if current_dir.name == "final_project":
    PROJECT_DIR = current_dir
elif (current_dir / "final_project").exists():
    PROJECT_DIR = current_dir / "final_project"
else:
    PROJECT_DIR = Path("/Users/gwongwangjae/goorm-ai-language-course/final_project")

DATA_PATH = PROJECT_DIR / "data" / "processed" / "transactions.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = PROJECT_DIR / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = "smoke"  # "smoke" or "full"
RANDOM_STATE = 42

SMOKE_LIMITS = {
    "train": 200_000,
    "valid": 50_000,
    "test": 50_000,
    "recent_holdout": 50_000,
}

BATCH_SIZE = 8192
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 4
REDUCE_LR_PATIENCE = 2
REDUCE_LR_FACTOR = 0.5
MIN_LR = 1e-5
LEARNING_RATE = 0.001

assert RUN_MODE in {"smoke", "full"}, "RUN_MODE must be either 'smoke' or 'full'."
assert DATA_PATH.exists(), f"transactions.csv not found: {DATA_PATH}"

tf.keras.utils.set_random_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_PATH:", DATA_PATH)
print("RUN_MODE:", RUN_MODE)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_DIR:", MODEL_DIR)

## 3. 데이터 로드와 Policy B 필터링

CSV는 모델링에 필요한 컬럼만 `usecols`로 읽습니다. 뉴스, 금리, 정책, 외부 시계열 변수는 사용하지 않습니다.

In [ ]:
# 2) 데이터 로드와 Policy B 필터링
USECOLS = [
    "transaction_id",
    "complex_id",
    "legal_dong_code",
    "sgg_code",
    "area_m2",
    "floor",
    "age_years",
    "deal_date",
    "trade_type",
    "is_cancelled",
    "price_total",
    "price_per_m2",
    "complex_prev_price_per_m2",
    "complex_prev_missing",
    "prev_deal_gap_days",
]

DTYPES = {
    "transaction_id": "string",
    "complex_id": "string",
    "legal_dong_code": "string",
    "sgg_code": "string",
    "area_m2": "float32",
    "floor": "float32",
    "age_years": "float32",
    "trade_type": "string",
    "is_cancelled": "Int8",
    "price_total": "float32",
    "price_per_m2": "float32",
    "complex_prev_price_per_m2": "float32",
    "complex_prev_missing": "Int8",
    "prev_deal_gap_days": "float32",
}

raw_df = pd.read_csv(DATA_PATH, usecols=USECOLS, dtype=DTYPES, parse_dates=["deal_date"])
raw_df["trade_type"] = raw_df["trade_type"].fillna("unknown")

policy_mask = (raw_df["is_cancelled"] == 0) & raw_df["trade_type"].isin(["중개거래", "unknown"])
df = raw_df.loc[policy_mask].copy()

assert len(df) > 0, "Policy B filtered data is empty."
print("raw rows:", len(raw_df))
print("policy_b rows:", len(df))
display(df.head())

## 4. Feature 생성과 Leakage 방지

정답 가격 컬럼과 거래 날짜/상태 컬럼은 모델 입력에서 제외합니다. 단지/법정동/구 ID는 이번 실험의 embedding 입력으로만 사용합니다.

In [ ]:
# 3) Feature 생성과 leakage 방지
NUMERIC_FEATURES = [
    "area_m2",
    "floor",
    "is_basement_floor",
    "age_years",
    "log_complex_prev_price_per_m2",
    "complex_prev_missing",
    "prev_deal_gap_months",
]

EMBEDDING_FEATURES = [
    "complex_id",
    "legal_dong_code",
    "sgg_code",
    "prev_deal_gap_bucket",
]

EMBEDDING_DIMS = {
    "complex_id": 32,
    "legal_dong_code": 16,
    "sgg_code": 8,
    "prev_deal_gap_bucket": 3,
}

FEATURES = NUMERIC_FEATURES + EMBEDDING_FEATURES

HARD_LEAKAGE_COLUMNS = {
    "target",
    "price_total",
    "price_per_m2",
    "reported_at",
    "deal_date",
    "deal_ym",
    "transaction_id",
    "raw_complex_name",
    "normalized_complex_name",
    "trade_type",
    "is_cancelled",
    "prev_price_ratio",
    "complex_prev_price_per_m2",
    "prev_deal_gap_days",
}
assert not (set(FEATURES) & HARD_LEAKAGE_COLUMNS), set(FEATURES) & HARD_LEAKAGE_COLUMNS


def add_features(input_df: pd.DataFrame) -> pd.DataFrame:
    out = input_df.copy()
    out["target"] = np.log(out["price_per_m2"].astype("float64"))
    out["is_basement_floor"] = (out["floor"] < 0).astype("float32")

    prev_price = out["complex_prev_price_per_m2"].astype("float64")
    out["log_complex_prev_price_per_m2"] = np.where(prev_price > 0, np.log(prev_price), np.nan)
    out["complex_prev_missing"] = out["complex_prev_missing"].fillna(1).astype("float32")
    out["prev_deal_gap_months"] = out["prev_deal_gap_days"].astype("float64") / 30.4375

    gap = out["prev_deal_gap_days"]
    bucket = pd.Series("missing", index=out.index, dtype="string")
    bucket[(gap >= 0) & (gap <= 30)] = "0-30"
    bucket[(gap >= 31) & (gap <= 90)] = "31-90"
    bucket[(gap >= 91) & (gap <= 180)] = "91-180"
    bucket[(gap >= 181) & (gap <= 365)] = "181-365"
    bucket[gap >= 366] = "366+"
    out["prev_deal_gap_bucket"] = bucket.fillna("missing")

    for feature in EMBEDDING_FEATURES:
        out[feature] = out[feature].fillna("missing").astype("string")
    return out


df = add_features(df)
assert df["target"].notna().all(), "target contains null values."
assert np.isfinite(df["target"]).all(), "target contains non-finite values."
assert (df["price_per_m2"] > 0).all(), "price_per_m2 must be positive."

display(df[FEATURES + ["target"]].head())

## 5. 시간 기준 Split과 Smoke Sampling

03 baseline과 동일한 시간 기준 split을 사용합니다.

- train: `deal_date <= 2023-12-31`
- valid: `2024-01-01 ~ 2024-12-31`
- test: `2025-01-01 ~ 2025-12-31`
- recent_holdout: `2026-01-01` 이후

In [ ]:
# 4) 시간 기준 split과 smoke sampling
SPLIT_ORDER = ["train", "valid", "test", "recent_holdout"]
SPLIT_DATES = {
    "train_end": "2023-12-31",
    "valid_start": "2024-01-01",
    "valid_end": "2024-12-31",
    "test_start": "2025-01-01",
    "test_end": "2025-12-31",
    "recent_holdout_start": "2026-01-01",
}


def split_frames(policy_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    splits = {
        "train": policy_df.loc[policy_df["deal_date"] <= SPLIT_DATES["train_end"]],
        "valid": policy_df.loc[(policy_df["deal_date"] >= SPLIT_DATES["valid_start"]) & (policy_df["deal_date"] <= SPLIT_DATES["valid_end"])],
        "test": policy_df.loc[(policy_df["deal_date"] >= SPLIT_DATES["test_start"]) & (policy_df["deal_date"] <= SPLIT_DATES["test_end"])],
        "recent_holdout": policy_df.loc[policy_df["deal_date"] >= SPLIT_DATES["recent_holdout_start"]],
    }
    for split_name, split_df in splits.items():
        assert len(split_df) > 0, f"{split_name} split is empty."
    return splits


def apply_smoke_sampling(splits: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    if RUN_MODE != "smoke":
        return {name: split_df.copy() for name, split_df in splits.items()}
    sampled = {}
    for split_name, split_df in splits.items():
        limit = SMOKE_LIMITS[split_name]
        if len(split_df) > limit:
            sampled[split_name] = split_df.sample(n=limit, random_state=RANDOM_STATE).sort_values("deal_date")
        else:
            sampled[split_name] = split_df.copy()
    return sampled


full_splits = split_frames(df)
run_splits = apply_smoke_sampling(full_splits)

counts_df = pd.DataFrame([
    {
        "policy": "policy_b_broker_unknown",
        "split": split_name,
        "full_rows": len(full_splits[split_name]),
        "run_rows": len(run_splits[split_name]),
    }
    for split_name in SPLIT_ORDER
])

display(counts_df)

## 6. Keras Preprocessing Layer Adapt

숫자형 median, `Normalization`, `StringLookup` vocabulary는 train split에만 맞춥니다. valid/test/recent 신규 ID는 OOV bucket으로 처리합니다.

In [ ]:
# 5) Keras preprocessing layer adapt
numeric_medians = run_splits["train"][NUMERIC_FEATURES].median(numeric_only=True).astype("float32")


def make_model_inputs(split_df: pd.DataFrame) -> dict[str, np.ndarray]:
    numeric_df = split_df[NUMERIC_FEATURES].copy()
    numeric_df = numeric_df.fillna(numeric_medians)
    inputs = {"numeric_input": numeric_df.to_numpy(dtype="float32")}
    for feature in EMBEDDING_FEATURES:
        values = np.asarray(split_df[feature].fillna("missing").astype("string").astype(str).tolist(), dtype=str).reshape(-1, 1)
        inputs[f"{feature}_input"] = tf.convert_to_tensor(values, dtype=tf.string)
    return inputs


def make_target(split_df: pd.DataFrame) -> np.ndarray:
    y = split_df["target"].to_numpy(dtype="float32")
    assert np.isfinite(y).all(), "target contains non-finite values."
    return y


train_inputs = make_model_inputs(run_splits["train"])
valid_inputs = make_model_inputs(run_splits["valid"])
y_train = make_target(run_splits["train"])
y_valid = make_target(run_splits["valid"])

normalizer = keras.layers.Normalization(name="numeric_normalization")
normalizer.adapt(train_inputs["numeric_input"])

lookup_layers = {}
for feature in EMBEDDING_FEATURES:
    lookup = keras.layers.StringLookup(num_oov_indices=1, mask_token=None, name=f"{feature}_lookup")
    lookup.adapt(train_inputs[f"{feature}_input"])
    lookup_layers[feature] = lookup


def unique_values(split_df: pd.DataFrame, feature: str) -> set[str]:
    return set(split_df[feature].fillna("missing").astype("string").astype(str).unique())


full_train_vocab = {feature: unique_values(full_splits["train"], feature) for feature in EMBEDDING_FEATURES}
run_train_vocab = {feature: unique_values(run_splits["train"], feature) for feature in EMBEDDING_FEATURES}

oov_rows = []
for feature in EMBEDDING_FEATURES:
    for split_name in SPLIT_ORDER:
        values = run_splits[split_name][feature].fillna("missing").astype("string").astype(str)
        is_oov = ~values.isin(run_train_vocab[feature])
        oov_rows.append({
            "feature": feature,
            "split": split_name,
            "full_train_unique": len(full_train_vocab[feature]),
            "run_train_unique": len(run_train_vocab[feature]),
            "lookup_vocabulary_size": lookup_layers[feature].vocabulary_size(),
            "rows": len(values),
            "oov_rows_vs_run_train": int(is_oov.sum()),
            "oov_rate_vs_run_train": float(is_oov.mean()),
        })

oov_df = pd.DataFrame(oov_rows)

for name, array in train_inputs.items():
    assert len(array) == len(run_splits["train"]), f"input length mismatch: {name}"

print("numeric input shape:", train_inputs["numeric_input"].shape)
display(oov_df)

## 7. 멀티-인풋 임베딩 모델 정의

In [ ]:
# 6) 멀티-인풋 임베딩 모델 정의
def build_embedding_model() -> keras.Model:
    numeric_input = keras.Input(shape=(len(NUMERIC_FEATURES),), name="numeric_input", dtype="float32")
    numeric_encoded = normalizer(numeric_input)

    model_inputs = [numeric_input]
    encoded_parts = [numeric_encoded]
    for feature in EMBEDDING_FEATURES:
        input_layer = keras.Input(shape=(1,), name=f"{feature}_input", dtype=tf.string)
        index = lookup_layers[feature](input_layer)
        embedding = keras.layers.Embedding(
            input_dim=lookup_layers[feature].vocabulary_size(),
            output_dim=EMBEDDING_DIMS[feature],
            name=f"{feature}_embedding",
        )(index)
        model_inputs.append(input_layer)
        encoded_parts.append(keras.layers.Flatten(name=f"{feature}_flatten")(embedding))

    x = keras.layers.Concatenate(name="feature_concat")(encoded_parts)
    x = keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-5), name="dense_128")(x)
    x = keras.layers.Dropout(0.10, name="dropout_010")(x)
    x = keras.layers.Dense(64, activation="relu", name="dense_64")(x)
    x = keras.layers.Dropout(0.05, name="dropout_005")(x)
    output = keras.layers.Dense(1, name="log_price_per_m2")(x)

    model = keras.Model(inputs=model_inputs, outputs=output, name="tf_multi_input_embedding")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model


model = build_embedding_model()
model.summary()

## 8. 학습과 Early Stopping

In [ ]:
# 7) 학습과 early stopping
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        patience=REDUCE_LR_PATIENCE,
        factor=REDUCE_LR_FACTOR,
        min_lr=MIN_LR,
    ),
]

start_time = time.perf_counter()
history = model.fit(
    train_inputs,
    y_train,
    validation_data=(valid_inputs, y_valid),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)
training_duration_seconds = time.perf_counter() - start_time

history_df = pd.DataFrame(history.history)
history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
best_epoch_index = int(history_df["val_loss"].idxmin())
best_epoch = int(history_df.loc[best_epoch_index, "epoch"])
best_valid_loss = float(history_df.loc[best_epoch_index, "val_loss"])
best_valid_mae = float(history_df.loc[best_epoch_index, "val_mae"])

training_info = {
    "epochs_ran": int(len(history_df)),
    "best_epoch": best_epoch,
    "best_valid_loss": best_valid_loss,
    "best_valid_mae": best_valid_mae,
    "training_duration_seconds": float(training_duration_seconds),
}

print(training_info)
display(history_df)

## 9. 평가와 Baseline 비교

03 baseline metrics가 있으면 Policy B의 LinearRegression/Ridge 중 valid `log_mae`가 가장 낮은 모델을 기준으로 비교합니다.

In [ ]:
# 8) 평가와 baseline 비교
BASELINE_METRICS_PATH = OUTPUT_DIR / "baseline_regression_metrics.csv"
METRICS_PATH = OUTPUT_DIR / "tf_multi_input_embedding_metrics.csv"
SUMMARY_PATH = OUTPUT_DIR / "tf_multi_input_embedding_summary.md"
PREDICTIONS_SAMPLE_PATH = OUTPUT_DIR / "tf_multi_input_embedding_predictions_sample.csv"
BEST_MODEL_PATH = MODEL_DIR / "tf_multi_input_embedding_best.keras"
RUN_CONFIG_PATH = MODEL_DIR / "tf_multi_input_embedding_run_config.json"


def load_baseline_metrics() -> tuple[pd.DataFrame | None, dict]:
    metadata = {
        "available": False,
        "path": str(BASELINE_METRICS_PATH),
        "baseline_policy": None,
        "baseline_model": None,
        "baseline_run_mode": None,
        "baseline_valid_log_mae": None,
    }
    if not BASELINE_METRICS_PATH.exists():
        return None, metadata

    baseline_df = pd.read_csv(BASELINE_METRICS_PATH)
    candidates = baseline_df.loc[
        (baseline_df["policy"] == "policy_b_broker_unknown")
        & (baseline_df["split"] == "valid")
        & (baseline_df["model"].isin(["linear_regression", "ridge_alpha_1"]))
    ].copy()
    if len(candidates) == 0:
        return baseline_df, metadata

    best = candidates.sort_values("log_mae").iloc[0]
    metadata.update({
        "available": True,
        "baseline_policy": str(best["policy"]),
        "baseline_model": str(best["model"]),
        "baseline_run_mode": str(best.get("run_mode", "unknown")),
        "baseline_valid_log_mae": float(best["log_mae"]),
    })
    return baseline_df, metadata


def baseline_log_mae_for_split(baseline_df: pd.DataFrame | None, metadata: dict, split_name: str) -> float:
    if baseline_df is None or not metadata["available"]:
        return np.nan
    rows = baseline_df.loc[
        (baseline_df["policy"] == metadata["baseline_policy"])
        & (baseline_df["model"] == metadata["baseline_model"])
        & (baseline_df["split"] == split_name)
    ]
    if len(rows) == 0:
        return np.nan
    return float(rows.iloc[0]["log_mae"])


def evaluate_predictions(split_df: pd.DataFrame, y_true: np.ndarray, y_pred: np.ndarray, split_name: str, baseline_df: pd.DataFrame | None, baseline_metadata: dict) -> dict:
    y_pred = np.asarray(y_pred, dtype="float64").reshape(-1)
    y_true = np.asarray(y_true, dtype="float64").reshape(-1)
    assert len(y_true) == len(y_pred) == len(split_df)

    pred_price_per_m2 = np.exp(y_pred)
    actual_price_per_m2 = split_df["price_per_m2"].to_numpy(dtype="float64")
    pred_total = pred_price_per_m2 * split_df["area_m2"].to_numpy(dtype="float64")
    actual_total = split_df["price_total"].to_numpy(dtype="float64")
    assert (pred_price_per_m2 > 0).all(), "exp(pred) must be positive."

    valid_mape_mask = actual_price_per_m2 > 0
    model_log_mae = float(mean_absolute_error(y_true, y_pred))
    baseline_log_mae = baseline_log_mae_for_split(baseline_df, baseline_metadata, split_name)
    delta_log_mae = model_log_mae - baseline_log_mae if np.isfinite(baseline_log_mae) else np.nan
    relative_improvement_pct = ((baseline_log_mae - model_log_mae) / baseline_log_mae * 100) if np.isfinite(baseline_log_mae) and baseline_log_mae != 0 else np.nan

    return {
        "run_mode": RUN_MODE,
        "policy": "policy_b_broker_unknown",
        "model": "tf_multi_input_embedding",
        "split": split_name,
        "rows": len(split_df),
        "log_mae": model_log_mae,
        "log_rmse": float(math.sqrt(mean_squared_error(y_true, y_pred))),
        "price_per_m2_mae": float(mean_absolute_error(actual_price_per_m2, pred_price_per_m2)),
        "price_per_m2_mape": float(np.mean(np.abs((actual_price_per_m2[valid_mape_mask] - pred_price_per_m2[valid_mape_mask]) / actual_price_per_m2[valid_mape_mask]))),
        "total_price_mae_manwon": float(mean_absolute_error(actual_total, pred_total)),
        "baseline_model": baseline_metadata.get("baseline_model"),
        "baseline_run_mode": baseline_metadata.get("baseline_run_mode"),
        "baseline_log_mae": baseline_log_mae,
        "delta_log_mae": delta_log_mae,
        "relative_improvement_pct": relative_improvement_pct,
        "beats_baseline": bool(delta_log_mae < 0) if np.isfinite(delta_log_mae) else False,
    }


def prediction_sample(split_df: pd.DataFrame, y_pred: np.ndarray, split_name: str, n: int = 200) -> pd.DataFrame:
    sample = split_df[["transaction_id", "deal_date", "complex_id", "legal_dong_code", "sgg_code", "area_m2", "price_total", "price_per_m2", "target"]].copy()
    sample["policy"] = "policy_b_broker_unknown"
    sample["model"] = "tf_multi_input_embedding"
    sample["split"] = split_name
    sample["pred_target"] = np.asarray(y_pred, dtype="float64").reshape(-1)
    sample["pred_price_per_m2"] = np.exp(sample["pred_target"])
    sample["pred_total"] = sample["pred_price_per_m2"] * sample["area_m2"].astype("float64")
    if len(sample) > n:
        sample = sample.sample(n=n, random_state=RANDOM_STATE)
    return sample.sort_values(["split", "deal_date", "transaction_id"])


baseline_df, baseline_metadata = load_baseline_metrics()
metrics_rows = []
prediction_samples = []

for split_name in SPLIT_ORDER:
    split_inputs = make_model_inputs(run_splits[split_name])
    y_split = make_target(run_splits[split_name])
    pred = model.predict(split_inputs, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
    metrics_rows.append(evaluate_predictions(run_splits[split_name], y_split, pred, split_name, baseline_df, baseline_metadata))
    if split_name in {"valid", "test", "recent_holdout"}:
        prediction_samples.append(prediction_sample(run_splits[split_name], pred, split_name, n=200))

metrics_df = pd.DataFrame(metrics_rows)
predictions_sample_df = pd.concat(prediction_samples, ignore_index=True)

print("baseline_metadata:", baseline_metadata)
display(metrics_df)

## 10. 결과 저장과 리뷰 문서 생성

In [ ]:
# 9) 결과 저장과 리뷰 문서 생성
def df_to_markdown(table_df: pd.DataFrame, floatfmt: str | None = None) -> str:
    formatted = table_df.copy()
    if floatfmt is not None:
        for col in formatted.select_dtypes(include=["float", "float32", "float64"]).columns:
            formatted[col] = formatted[col].map(lambda value: format(value, floatfmt) if pd.notna(value) else "")
    formatted = formatted.astype("string").fillna("")
    headers = list(formatted.columns)
    rows = formatted.values.tolist()
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        lines.append("| " + " | ".join(str(value) for value in row) + " |")
    return "\n".join(lines)


def metric_value(split_name: str, column: str) -> float:
    rows = metrics_df.loc[metrics_df["split"] == split_name, column]
    return float(rows.iloc[0]) if len(rows) else float("nan")


metrics_df.to_csv(METRICS_PATH, index=False)
predictions_sample_df.to_csv(PREDICTIONS_SAMPLE_PATH, index=False)

valid_row = metrics_df.loc[metrics_df["split"] == "valid"].iloc[0]
test_row = metrics_df.loc[metrics_df["split"] == "test"].iloc[0]
recent_row = metrics_df.loc[metrics_df["split"] == "recent_holdout"].iloc[0]

valid_beats = bool(valid_row["beats_baseline"])
test_beats = bool(test_row["beats_baseline"])
recent_delta_vs_test = metric_value("recent_holdout", "log_mae") - metric_value("test", "log_mae")
recent_collapse = bool(recent_delta_vs_test > 0.03)
valid_mape_pct = metric_value("valid", "price_per_m2_mape") * 100

oov_complex_valid = oov_df.loc[(oov_df["feature"] == "complex_id") & (oov_df["split"] == "valid"), "oov_rate_vs_run_train"].iloc[0]
oov_complex_test = oov_df.loc[(oov_df["feature"] == "complex_id") & (oov_df["split"] == "test"), "oov_rate_vs_run_train"].iloc[0]
oov_complex_recent = oov_df.loc[(oov_df["feature"] == "complex_id") & (oov_df["split"] == "recent_holdout"), "oov_rate_vs_run_train"].iloc[0]

if baseline_metadata["available"]:
    baseline_sentence = (
        f"03 baseline은 `{baseline_metadata['baseline_run_mode']}` run의 "
        f"Policy B `{baseline_metadata['baseline_model']}`를 사용했다. "
        f"valid log_mae 기준값은 `{baseline_metadata['baseline_valid_log_mae']:.6f}`이다."
    )
else:
    baseline_sentence = "03 baseline metrics를 찾지 못해 baseline 비교 컬럼은 비워 두었다."

if valid_beats:
    valid_review = "valid log_mae가 baseline보다 낮아져 단지/지역 embedding이 기존 수치형 feature에서 설명하지 못한 지역성과 단지별 가격 성향을 일부 학습한 것으로 해석한다."
else:
    valid_review = "valid log_mae가 baseline보다 낮아지지 않아 현재 구조에서는 선형 baseline 대비 embedding 모델의 1차 성능 개선을 확인하지 못했다."

if valid_beats and not test_beats:
    generalization_review = "valid는 개선됐지만 test에서 개선이 유지되지 않아 embedding이 train/valid에 과적합됐거나 신규 단지 OOV 처리 영향이 있었을 가능성을 검토해야 한다."
elif test_beats:
    generalization_review = "test에서도 baseline 대비 개선이 유지되어 valid 개선이 단순한 split 우연만은 아닐 가능성이 있다."
else:
    generalization_review = "test에서도 baseline보다 나아지지 않아 직전 거래 단가 feature가 이미 강한 설명력을 가진 것으로 해석한다."

if recent_collapse:
    recent_review = f"recent_holdout log_mae가 test보다 `{recent_delta_vs_test:.6f}` 높아져 최근 기간에서 성능 저하가 뚜렷하다."
else:
    recent_review = f"recent_holdout log_mae와 test log_mae의 차이는 `{recent_delta_vs_test:.6f}`로, test 대비 급격한 붕괴 신호는 제한적이다."

if baseline_metadata["available"] and valid_beats and test_beats and not recent_collapse:
    outcome = "성공"
elif baseline_metadata["available"] and valid_beats:
    outcome = "부분 성공"
else:
    outcome = "실패 또는 보류"

summary_lines = []
summary_lines.append("# TF Multi-Input Embedding 결과 보고서")
summary_lines.append("")
summary_lines.append("## 1. 실험 목적")
summary_lines.append("`transactions.csv` 내부 정보만 사용해 단지/법정동/구 ID embedding이 03 선형 baseline보다 가격 예측 성능을 개선하는지 검증했다.")
summary_lines.append("")
summary_lines.append("## 2. 사용 데이터와 split")
summary_lines.append(f"- run_mode: `{RUN_MODE}`")
summary_lines.append(f"- random_state: `{RANDOM_STATE}`")
summary_lines.append("- policy: `is_cancelled == 0` and `trade_type in [중개거래, unknown]`")
summary_lines.append("- target: `log(price_per_m2)`")
summary_lines.append(df_to_markdown(counts_df))
summary_lines.append("")
summary_lines.append("## 3. 모델 구조")
summary_lines.append("숫자형 feature는 train median 보정 후 Keras `Normalization`을 train split에만 adapt했다. ID feature는 `StringLookup(num_oov_indices=1, mask_token=None)`와 embedding을 사용했다.")
summary_lines.append("")
summary_lines.append(df_to_markdown(pd.DataFrame([{"feature": key, "embedding_dim": value, "lookup_vocabulary_size": lookup_layers[key].vocabulary_size()} for key, value in EMBEDDING_DIMS.items()])))
summary_lines.append("")
summary_lines.append("네트워크는 numeric branch와 4개 embedding branch를 concat한 뒤 `Dense(128, relu, l2=1e-5) -> Dropout(0.10) -> Dense(64, relu) -> Dropout(0.05) -> Dense(1)` 구조로 학습했다.")
summary_lines.append("")
summary_lines.append("## 4. Baseline 비교")
summary_lines.append(baseline_sentence)
summary_lines.append(df_to_markdown(metrics_df[["split", "log_mae", "baseline_model", "baseline_run_mode", "baseline_log_mae", "delta_log_mae", "relative_improvement_pct", "beats_baseline"]], floatfmt=".6f"))
summary_lines.append("")
summary_lines.append("## 5. TF Embedding 학습 결과")
summary_lines.append(f"- epochs_ran: `{training_info['epochs_ran']}`")
summary_lines.append(f"- best_epoch: `{training_info['best_epoch']}`")
summary_lines.append(f"- best_valid_loss: `{training_info['best_valid_loss']:.6f}`")
summary_lines.append(f"- best_valid_mae: `{training_info['best_valid_mae']:.6f}`")
summary_lines.append(f"- training_duration_seconds: `{training_info['training_duration_seconds']:.2f}`")
summary_lines.append(df_to_markdown(metrics_df, floatfmt=".6f"))
summary_lines.append("")
summary_lines.append("## 6. 결과 리뷰")
summary_lines.append(f"- valid 기준 baseline 승리 여부: `{valid_beats}`")
summary_lines.append(f"- test 기준 개선 유지 여부: `{test_beats}`")
summary_lines.append(f"- recent_holdout 성능 붕괴 여부: `{recent_collapse}`")
summary_lines.append(f"- valid 단가 기준 평균 오차율: `{valid_mape_pct:.2f}%`")
summary_lines.append(f"- 실험 판정: `{outcome}`")
summary_lines.append(valid_review)
summary_lines.append(generalization_review)
summary_lines.append(recent_review)
summary_lines.append(f"complex_id OOV rate는 valid `{oov_complex_valid:.4%}`, test `{oov_complex_test:.4%}`, recent_holdout `{oov_complex_recent:.4%}`로 관측됐다. 신규 단지 OOV는 embedding 모델 성능에 영향을 줬을 가능성이 있다.")
summary_lines.append("MLP 계열 embedding 모델의 실질적 이득은 valid/test/recent의 baseline 대비 개선이 함께 유지되는지로 판단한다.")
summary_lines.append("")
summary_lines.append("## 7. 개선 실패 또는 성공 원인 해석")
if outcome == "성공":
    summary_lines.append("valid와 test에서 개선이 유지된다면 단지/지역 embedding이 수치형 feature에 없는 위치성과 단지별 가격 성향을 보완한 것으로 해석한다.")
elif outcome == "부분 성공":
    summary_lines.append("valid는 개선됐지만 test/recent가 악화됐다면 embedding이 train/valid에 과적합됐거나 신규 complex_id OOV 처리 영향이 있었을 가능성을 검토한다.")
else:
    summary_lines.append("baseline보다 개선되지 않았다면 현재 직전 거래 단가 feature가 이미 강한 설명력을 가지고 있어 단지/지역 embedding의 추가 이득이 제한적이었다고 해석한다.")
summary_lines.append("")
summary_lines.append("## 8. 다음 발전 방향")
summary_lines.append("- 성공 시: embedding dimension 튜닝, Dense layer depth 튜닝, rare bucket 처리, 지역 interaction 추가, full run 반복 seed 검증")
summary_lines.append("- 부분 성공 시: 과적합 여부, OOV 비율, 단지별 거래 수 편차, Dropout/L2 강화, rare complex_id 통합, embedding dimension 축소 점검")
summary_lines.append("- 실패 시: Ridge baseline 공식 기준 유지, target encoding 실험, LightGBM/XGBoost 별도 검토, 지역/단지 feature 품질 점검")
summary_lines.append("")
summary_lines.append("## 9. 생성 산출물")
summary_lines.append(f"- metrics: `{METRICS_PATH}`")
summary_lines.append(f"- summary: `{SUMMARY_PATH}`")
summary_lines.append(f"- prediction_sample: `{PREDICTIONS_SAMPLE_PATH}`")
if RUN_MODE == "full":
    summary_lines.append(f"- best_model: `{BEST_MODEL_PATH}`")
    summary_lines.append(f"- run_config: `{RUN_CONFIG_PATH}`")

SUMMARY_PATH.write_text("\n".join(summary_lines), encoding="utf-8")

if RUN_MODE == "full":
    model.save(BEST_MODEL_PATH)
    run_config = {
        "run_mode": RUN_MODE,
        "random_state": RANDOM_STATE,
        "policy": "Policy B: is_cancelled == 0 and trade_type in [중개거래, unknown]",
        "split_dates": SPLIT_DATES,
        "numeric_features": NUMERIC_FEATURES,
        "embedding_features": EMBEDDING_FEATURES,
        "embedding_dims": EMBEDDING_DIMS,
        "vocab_sizes": {feature: int(lookup_layers[feature].vocabulary_size()) for feature in EMBEDDING_FEATURES},
        "oov_counts_by_split": oov_df.to_dict(orient="records"),
        "training_config": {
            "optimizer": "Adam",
            "learning_rate": LEARNING_RATE,
            "loss": "mse",
            "metric": "mae",
            "batch_size": BATCH_SIZE,
            "max_epochs": MAX_EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "reduce_lr_patience": REDUCE_LR_PATIENCE,
            "reduce_lr_factor": REDUCE_LR_FACTOR,
            "min_lr": MIN_LR,
        },
        "baseline_metrics_used": baseline_metadata,
        "best_epoch": best_epoch,
        "training_info": training_info,
        "final_metrics": metrics_df.to_dict(orient="records"),
    }
    RUN_CONFIG_PATH.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")

assert METRICS_PATH.exists(), "metrics output was not created."
assert SUMMARY_PATH.exists(), "summary output was not created."
assert PREDICTIONS_SAMPLE_PATH.exists(), "prediction sample output was not created."
print("saved:", METRICS_PATH)
print("saved:", SUMMARY_PATH)
print("saved:", PREDICTIONS_SAMPLE_PATH)
if RUN_MODE == "full":
    print("saved:", BEST_MODEL_PATH)
    print("saved:", RUN_CONFIG_PATH)